In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

load_dotenv(override=True)

key = os.getenv("OPENROUTER_API_KEY")
if key:
    print("API key loaded!")
else:
    print("Key not found - check .env file!")

print("All imports done!")

API key loaded!
All imports done!


In [3]:
loader = PyPDFLoader("Tech_and_AI_Report_2025.pdf")
pages = loader.load()
print(f"PDF loaded! Pages: {len(pages)}")

PDF loaded! Pages: 20


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = splitter.split_documents(pages)
print(f"Chunks created: {len(splits)}")

Chunks created: 74


In [5]:
print("Loading embeddings... please wait...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
print("Embeddings ready!")

Loading embeddings... please wait...


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 0cae7439-b8aa-415d-ae90-f1c4a518c640)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 760af307-1328-41ca-8e21-be811a48455c)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json
Retrying in 2s [Retry 2/5].


Embeddings ready!


In [6]:
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)
print("Vector database ready!")

Vector database ready!


In [7]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)
print("LLM ready!")

LLM ready!


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)
print("Q/A Chain ready! Ask your questions below!")

Q/A Chain ready! Ask your questions below!


In [1]:
query = "Define Cybersecurity in the AI Era"
response = qa_chain.invoke(query)
print("Question:", query)
print("\nAnswer:", response["result"])

NameError: name 'qa_chain' is not defined